# Sinkhorn balancing on an uneven test: a risk simulator

Most public notebooks end with **Sinkhorn balancing**: test probabilities are rescaled until every
model gets the same total mass. That is a sensible correction for balanced accuracy *if the
hidden test is close to even*. Nobody outside the hosts knows the test's mix. This notebook
measures what balancing does to a given model's predictions when the test is even and when it is
not.

**The answer depends on the model.** For the TF-IDF baseline below, balancing loses at almost
every test mix. For a strong blend (about 0.64 balanced accuracy, numbers in the discussion
linked below), it gained a little on an even test and lost a lot on an uneven one. So it is worth
checking your own OOF predictions before adding Sinkhorn to a submission. Replace `oof` in
section 1 to do that.

The idea:

1. Build out-of-fold (OOF) probabilities with prompt-grouped 5-fold CV. Here that is a plain
   TF-IDF + logistic regression baseline, so the notebook runs on CPU in a few minutes. Swap in
   your own OOF file to test your own blend.
2. Draw test pools whose model mix follows a symmetric **Dirichlet(alpha)**: alpha = 1 is very
   uneven, with some models nearly absent; alpha = 100 is nearly even.
3. On each pool, score the plain argmax and Sinkhorn with 1, 3, 10 and 50 iterations.

Discussion with numbers for a stronger blend (about 0.64 balanced accuracy):
https://www.kaggle.com/competitions/name-that-llm/discussion/752365

## 1. OOF probabilities from a public baseline

In [ ]:
import numpy as np, pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold

train = pd.read_csv("/kaggle/input/competitions/name-that-llm/train.csv")
labels = np.array(sorted(train.model.unique()))
y = train.model.to_numpy()
oof = np.zeros((len(train), len(labels)))
views = [dict(analyzer="char", ngram_range=(1, 5), max_features=200_000),
         dict(analyzer="word", ngram_range=(1, 2), max_features=100_000, token_pattern=r"(?u)\b\w+\b|[^\w\s]")]
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for k, (tr, va) in enumerate(cv.split(train, y, groups=train.prompt_id)):
    vecs = [TfidfVectorizer(sublinear_tf=True, lowercase=False, min_df=2, **v) for v in views]
    Xtr = hstack([v.fit_transform(train.text.iloc[tr]) for v in vecs]).tocsr()
    Xva = hstack([v.transform(train.text.iloc[va]) for v in vecs]).tocsr()
    clf = LogisticRegression(C=10, max_iter=2000).fit(Xtr, y[tr])
    oof[va] = clf.predict_proba(Xva)[:, [list(clf.classes_).index(c) for c in labels]]
    print(f"fold {k}: {balanced_accuracy_score(y[va], labels[oof[va].argmax(1)]):.4f}")
logp = np.log(np.clip(oof, 1e-12, None))
print(f"OOF balanced accuracy: {balanced_accuracy_score(y, labels[logp.argmax(1)]):.4f}")

## 2. Balancing, and test pools with an uneven mix

In [ ]:
def sinkhorn(logp, iters):
    """Rescale log-probabilities until every class gets the same total mass; rows stay normalised."""
    z, target = logp.copy(), np.log(len(logp) / logp.shape[1])
    for _ in range(iters):
        m = z.max(0, keepdims=True)
        z = z - (m + np.log(np.exp(z - m).sum(0, keepdims=True))) + target
        z = z - z.max(1, keepdims=True)
        z -= np.log(np.exp(z).sum(1, keepdims=True))
    return z

def uneven_pools(y, alpha, size=2000, draws=40, seed=0):
    """Row indices of test pools whose class mix is drawn from a symmetric Dirichlet(alpha)."""
    rng, classes = np.random.default_rng(seed), np.unique(y)
    by_class = [np.flatnonzero(y == c) for c in classes]
    for _ in range(draws):
        counts = rng.multinomial(size, rng.dirichlet([alpha] * len(classes)))
        yield np.concatenate([rng.choice(ix, k) for ix, k in zip(by_class, counts) if k])

ITERS = [0, 1, 3, 10, 50]
COLS = ["plain" if k == 0 else f"sinkhorn_{k}" for k in ITERS]
def score(rows):
    return [balanced_accuracy_score(y[rows], labels[sinkhorn(logp[rows], k).argmax(1)]) for k in ITERS]

import warnings; warnings.filterwarnings("ignore")   # pools without some models warn in sklearn
ALPHAS = [0.5, 1, 2, 5, 10, 20, 50, 100]
rows = [{"mix": "even (all OOF rows)", "alpha": np.inf, **dict(zip(COLS, score(np.arange(len(y)))))}]
for a in ALPHAS:
    rows.append({"mix": f"Dirichlet({a})", "alpha": a, **dict(zip(COLS, np.mean([score(r) for r in uneven_pools(y, a)], 0)))})
res = pd.DataFrame(rows)
res.round(4)

## 3. Gain or loss against the plain argmax

In [ ]:
import matplotlib.pyplot as plt

SERIES = [("sinkhorn_1", "1 iteration", "#2a78d6"), ("sinkhorn_3", "3 iterations", "#eb6834"),
          ("sinkhorn_10", "10 iterations", "#1baf7a"), ("sinkhorn_50", "50 iterations", "#eda100")]
d = res[np.isfinite(res.alpha)]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhline(0, color="#8a8a85", lw=1, zorder=1)
for col, name, color in SERIES:
    delta = d[col] - d.plain
    ax.plot(d.alpha, delta, color=color, lw=2, marker="o", ms=5, label=name, zorder=3)
ax.set_xscale("log")
ax.set_xticks(ALPHAS, [str(a) for a in ALPHAS])
ax.set_xlabel("Dirichlet alpha of the test mix (left: very uneven, right: nearly even)")
ax.set_ylabel("balanced accuracy, minus plain argmax")
ax.set_title("Sinkhorn balancing minus plain argmax, by test mix (TF-IDF baseline)", loc="left", fontsize=11)
ax.grid(axis="y", color="#e6e6e3", lw=0.8); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.legend(frameon=False, loc="lower right", fontsize=9)
ax.set_xlim(0.4, 130)
plt.tight_layout(); plt.show()

## 4. Pool size

The same comparison at alpha = 5, with different pool sizes. A pool of 75 passages is close to
what a sparse CV fold looks like: 1 to 4 passages per prompt, with some models absent.

In [ ]:
sizes = [75, 300, 1000, 4000]
rows = []
for n in sizes:
    m = np.mean([score(r) for r in uneven_pools(y, 5, size=n, draws=60)], 0)
    rows.append({"pool": n, **dict(zip(COLS, m))})
by_size = pd.DataFrame(rows)
by_size.round(4)

## What to take from it

- **For this TF-IDF + logistic regression baseline, balancing does not pay.** It loses about
  0.003 on an exactly even test and up to 0.014 on a very uneven one. Only on the smallest
  pools (75 passages) does one iteration gain a little.
- **For a strong blend it was different** (see the discussion): one to three iterations gained
  about 0.002 on an even test, and 50 iterations lost up to 0.066 on a very uneven one.
- **More iterations cost more when the mix is uneven.** If you do balance, a few iterations are a
  cheaper bet than running Sinkhorn to convergence.
- **OOF predictions only show the risk.** They carry no shift between train and test. If
  balancing helps you on the public LB, that tells you your test predictions lean towards some
  models, not what the test's mix is.

To test your own blend, replace `oof` in section 1 with your OOF probabilities: rows in train
order, columns in `labels` order.